# 04 · Single-agent LLM prompting (S1 zero-shot, S2 few-shot, S3 chain-of-thought)

Runs every model × setting × paraphrase (× seed for S2) through Ollama with JSON-constrained output. Each prediction is appended to a JSONL file in Google Drive, so the notebook **resumes** after a Colab disconnect—just run it again. Tip: set `limit_test_samples: 50` in the config for a pilot first.

In [1]:
# ==== Setup (run first) ====
# SOURCE = "drive"  : the repository folder is on Google Drive (use this now)
# SOURCE = "github" : clone from GitHub (after the repository is public)
SOURCE = "drive"
DRIVE_REPO = "/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment"   # folder from the unzipped package
REPO_URL = "https://github.com/deepittalk-repo/mad-indo-sentiment.git"   # used only when SOURCE = "github"

import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # results persist in MyDrive/mad-indo-results
    if SOURCE == "drive":
        REPO_DIR = DRIVE_REPO
        assert os.path.exists(REPO_DIR), f"Folder not found: {REPO_DIR}. Upload the unzipped folder to MyDrive first."
    else:
        REPO_DIR = "/content/mad-indo-sentiment"
        if not os.path.exists(REPO_DIR):
            !git clone -q $REPO_URL $REPO_DIR
    %cd $REPO_DIR
    !pip -q install -r requirements.txt
else:
    # Local run: start Jupyter from the repository root or the notebooks/ folder.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, "src")

from mad_indo import data
cfg = data.load_config()
RESULTS = data.results_dir()
print("Results folder:", RESULTS)
LIMIT = cfg["limit_test_samples"]   # null in config = full test set


Mounted at /content/drive
/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment
Results folder: /content/drive/MyDrive/mad-indo-results


In [2]:
from mad_indo.llm import setup_ollama, OllamaClient
lc = cfg["llm"]
setup_ollama(list(lc["models"].values()), host=lc["host"], num_parallel=lc["parallel_requests"])
client = OllamaClient(lc["host"], num_ctx=lc["num_ctx"], num_predict=lc["num_predict"])

Installing Ollama ...
Starting Ollama server (parallel=4) ...
Pulling qwen2.5:7b ...
Pulling llama3.1:8b ...
Pulling gemma2:9b ...
Ollama ready | ollama version is 0.40.16 | models: qwen2.5:7b, llama3.1:8b, gemma2:9b


Choose what to run in this session (a full run is long; split it across sessions if needed).

In [3]:
RUN_TASKS = ["smsa", "emot"]
RUN_MODELS = list(lc["models"].keys())          # e.g. ["qwen"] to run one family per session
RUN_SETTINGS = cfg["single_agent"]["settings"]

In [4]:
from mad_indo.runners import run_single_agent
out_dir = RESULTS / "single_agent"; out_dir.mkdir(exist_ok=True)
sa = cfg["single_agent"]
for task in RUN_TASKS:
    L = cfg["tasks"][task]["labels"]
    train = data.load_split(task, "train"); test = data.load_split(task, "test", limit=LIMIT)
    for mk in RUN_MODELS:
        for setting in RUN_SETTINGS:
            seeds = sa["seeds"] if setting == "S2_few_shot" else [sa["seeds"][0]]
            paraphrases = range(sa["n_paraphrases"]) if mk in sa["paraphrase_models"] else [0]
            for p in paraphrases:
                for seed in seeds:
                    print(f"{task} | {mk} | {setting} | paraphrase {p} | seed {seed}")
                    run_single_agent(client, mk, lc["models"][mk], task, L, test, train,
                                     setting, p, seed, cfg, out_dir / f"{task}_{mk}.jsonl",
                                     workers=lc["parallel_requests"])

smsa | qwen | S1_zero_shot | paraphrase 0 | seed 11
smsa | qwen | S1_zero_shot | paraphrase 1 | seed 11
smsa | qwen | S1_zero_shot | paraphrase 2 | seed 11
smsa | qwen | S2_few_shot | paraphrase 0 | seed 11
smsa | qwen | S2_few_shot | paraphrase 0 | seed 22
smsa | qwen | S2_few_shot | paraphrase 0 | seed 33
smsa | qwen | S2_few_shot | paraphrase 1 | seed 11
smsa | qwen | S2_few_shot | paraphrase 1 | seed 22
smsa | qwen | S2_few_shot | paraphrase 1 | seed 33
smsa | qwen | S2_few_shot | paraphrase 2 | seed 11
smsa | qwen | S2_few_shot | paraphrase 2 | seed 22
smsa | qwen | S2_few_shot | paraphrase 2 | seed 33
smsa | qwen | S3_cot | paraphrase 0 | seed 11
smsa | qwen | S3_cot | paraphrase 1 | seed 11
smsa | qwen | S3_cot | paraphrase 2 | seed 11
smsa | llama | S1_zero_shot | paraphrase 0 | seed 11
smsa | llama | S2_few_shot | paraphrase 0 | seed 11
smsa | llama | S2_few_shot | paraphrase 0 | seed 22
smsa | llama | S2_few_shot | paraphrase 0 | seed 33
smsa | llama | S3_cot | paraphrase 0 |